# 02 — Pandas Basics (ARIS Phase 1)

**Purpose:** prove pandas fluency before Phase 2 ingest work. The lap-time predictor and the stint-analysis notebook both lean hard on groupby/merge/resample.

**Self-test target (from `SKILLS-MASTERY.md` §2.2):** Series vs DataFrame, indexing (`.loc` / `.iloc`), filtering, groupby-aggregate, merge, time-series resample, missing-data handling.

Run top-to-bottom on a fresh kernel. Clear all outputs before committing.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=42)
print("pandas", pd.__version__)

## 1. Series — a 1-D array with an index

Like a NumPy 1-D array, but every element has a label (the index). The index is the thing that makes pandas different from NumPy — it's what `groupby`, `merge`, and `resample` all use under the hood.

In [ ]:
lap_times = pd.Series(
    [92.4, 92.2, 92.0, 91.8, 91.9],
    index=["L1", "L2", "L3", "L4", "L5"],
    name="VER_lap_time",
)
print(lap_times)
print("---")
print("mean:", lap_times.mean(), "| min:", lap_times.min(), "| L3 by label:", lap_times["L3"])

## 2. DataFrame — a 2-D table with row + column indices

The workhorse. Each column is a Series; they share a single row index. This is the shape `session.laps` returns from FastF1.

In [ ]:
laps = pd.DataFrame({
    "driver":    ["VER", "VER", "VER", "HAM", "HAM", "HAM", "LEC", "LEC", "LEC"],
    "lap":       [1, 2, 3, 1, 2, 3, 1, 2, 3],
    "compound":  ["SOFT", "SOFT", "SOFT", "MEDIUM", "MEDIUM", "MEDIUM", "SOFT", "SOFT", "SOFT"],
    "lap_time":  [92.4, 92.2, 92.0, 93.1, 92.9, 92.8, 92.6, 92.4, 92.3],
    "tyre_life": [5, 6, 7, 12, 13, 14, 3, 4, 5],
})
print("shape:", laps.shape, "| dtypes:")
print(laps.dtypes)
laps

## 3. Selection — `.loc` (by label) vs `.iloc` (by position)

Pick one and remember which. Mixing them is the most common pandas bug.

- `df.loc[row_label, col_label]` — labels, end-inclusive on slices
- `df.iloc[row_pos, col_pos]` — integer positions, end-exclusive (NumPy-style)

In [ ]:
print("by column:")
print(laps["lap_time"].head(3))
print("\n.loc — first 3 rows, two named cols (end-inclusive):")
print(laps.loc[0:2, ["driver", "lap_time"]])
print("\n.iloc — same rows by position, last 2 cols (end-exclusive):")
print(laps.iloc[0:3, -2:])

## 4. Boolean filtering

Same idea as NumPy boolean indexing; combine masks with `&`, `|`, `~` (not `and`/`or`/`not`) and parenthesise every clause.

In [ ]:
fast_softs = laps[(laps["compound"] == "SOFT") & (laps["lap_time"] < 92.5)]
print(fast_softs)

## 5. groupby — split / apply / combine

The single most-used pandas operation in ARIS. `groupby` makes one bucket per unique key; you then run an aggregation (`mean`, `min`, `count`, …) or a custom function on each bucket.

In [ ]:
# mean lap time per driver
per_driver = laps.groupby("driver")["lap_time"].mean()
print("per-driver mean:\n", per_driver, "\n")

# multi-key + multi-agg — best/worst lap per (driver, compound)
agg = (
    laps.groupby(["driver", "compound"])
        .agg(best=("lap_time", "min"), worst=("lap_time", "max"), n=("lap_time", "size"))
        .reset_index()
)
agg

## 6. merge — SQL-style joins

When two tables share a key. `how` controls which rows survive (`inner`, `left`, `right`, `outer`). Default is `inner` — drops keys that aren't in both tables.

In [ ]:
drivers = pd.DataFrame({
    "driver": ["VER", "HAM", "LEC", "NOR"],   # NOR has no laps in `laps`
    "team":   ["Red Bull", "Mercedes", "Ferrari", "McLaren"],
})

merged = laps.merge(drivers, on="driver", how="left")
print("left-merged shape:", merged.shape)   # same row count as laps
merged.head()

## 7. Missing data

FastF1 lap frames have plenty of NaNs (in-laps, out-laps, deleted laps, sector times missing on virtual safety cars). Three habits:
- `isna().sum()` to find the hot spots
- `dropna(subset=[...])` to drop rows missing a *specific* column you need
- `fillna(...)` only when the fill value is defensible — never silently

In [ ]:
dirty = laps.copy()
dirty.loc[[1, 5], "lap_time"] = np.nan      # pretend two laps were deleted
print("missing per col:")
print(dirty.isna().sum(), "\n")

clean = dirty.dropna(subset=["lap_time"])
print("after dropna:", clean.shape)

## 8. Time-series — datetime index + `resample`

Telemetry samples are timestamped at ~3–10 Hz. To downsample to a regular cadence (say 1 Hz for plotting), set a datetime index and call `resample(rule).mean()` — like a groupby but with time bins.

In [ ]:
# fake 5 Hz speed trace over 4 seconds
ts = pd.date_range("2024-03-02 15:00:00", periods=20, freq="200ms")
speed = 280 + 20 * np.sin(np.linspace(0, np.pi, 20)) + rng.normal(0, 0.5, 20)
tele = pd.DataFrame({"speed_kmh": speed}, index=ts)

downsampled = tele.resample("1s").mean()
print("raw 5 Hz:  ", tele.shape, "first 3:\n", tele.head(3), "\n")
print("1 Hz mean: ", downsampled.shape, "\n", downsampled)

## 9. ARIS-flavoured exercise — fuel-corrected pace per driver

Given a lap frame with `lap`, `lap_time`, and a per-lap `fuel_correction`, compute each driver's mean *fuel-corrected* lap time, sorted fastest first. One pipeline, no Python loops.

In [ ]:
fuel_correction = pd.Series(
    [0.0, -0.05, -0.10],            # heavier early laps run slower; subtract that benefit
    index=[1, 2, 3],                # indexed by lap number
    name="fuel_correction",
)

ranking = (
    laps.assign(corrected=laps["lap_time"] + laps["lap"].map(fuel_correction))
        .groupby("driver")["corrected"]
        .mean()
        .sort_values()
)
print("fuel-corrected pace ranking (fastest first):\n", ranking)

## Takeaway

If §1–§8 ran clean and the §9 pipeline read naturally, the Phase 1 pandas bar is met:
*"can wrangle a FastF1 lap frame end-to-end without reaching for raw Python loops"*.

Edge cases left for later:
- `MultiIndex` (groupby + `unstack`) — needed in Phase 3 for per-driver/per-compound feature tables
- `pd.Categorical` for low-cardinality cols (`Compound`, `Team`) — model-training speedup
- `merge_asof` for joining telemetry to laps on the nearest timestamp — Phase 3+

Next: stint-analysis notebook (`03-stint-analysis.ipynb`) — first ARIS analysis that actually says something about a race.